In [1]:
from pyspark.sql import SparkSession

# Cấu hình
spark = SparkSession.builder \
    .appName("Iceberg-Fix-Final") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://rest:8181") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "mypassword") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Tạo Namespace (Namespace có thể coi như Database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.my_namespace")

# Tạo Table
spark.sql("""
    CREATE TABLE IF NOT EXISTS my_catalog.my_namespace.my_table (
        id bigint,
        name string,
        department string
    ) USING iceberg
""")

# Kiểm tra Spark chạy ổn không (nếu kết quả là một bảng trống thì Spark đã hoạt động tốt)
spark.sql("SELECT * FROM my_catalog.my_namespace.my_table").show()

26/05/04 11:31:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+---+----+----------+
| id|name|department|
+---+----+----------+
+---+----+----------+



In [2]:
# Chèn dữ liệu
spark.sql("INSERT INTO my_catalog.my_namespace.my_table VALUES (1, 'Trung', 'DE'), (2, 'Tuan', 'BA')")

# Kiểm tra kết quả
spark.sql("SELECT * FROM my_catalog.my_namespace.my_table").show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Trung|        DE|
|  2| Tuan|        BA|
+---+-----+----------+



In [3]:
# Đây là các lệnh xóa Table và Namespace (bạn phải xóa Table trước, xóa Namespace sau, không làm ngược lại)

# spark.sql("DROP TABLE my_catalog.my_namespace.my_table PURGE")
# spark.sql("DROP NAMESPACE my_catalog.my_namespace")

In [4]:
print("--- Lịch sử (Snapshots) ---")
spark.sql("""
          SELECT committed_at, snapshot_id, operation
          FROM my_catalog.my_namespace.my_table.snapshots""").show()

--- Lịch sử (Snapshots) ---
+--------------------+-------------------+---------+
|        committed_at|        snapshot_id|operation|
+--------------------+-------------------+---------+
|2026-05-04 11:31:...|7628884848170549472|   append|
+--------------------+-------------------+---------+



In [5]:
# Chèn dữ liệu
spark.sql("INSERT INTO my_catalog.my_namespace.my_table VALUES (3, 'Duy', 'AI'), (4, 'Thao', 'DA')")
spark.sql("INSERT INTO my_catalog.my_namespace.my_table VALUES (5, 'ChatGPT', 'LLM')")

# Xóa dữ liệu
spark.sql("DELETE FROM my_catalog.my_namespace.my_table WHERE id = 5")

# Chèn dữ liệu
spark.sql("INSERT INTO my_catalog.my_namespace.my_table VALUES (6, 'Gemini', 'LLM')")

DataFrame[]

In [6]:
print("--- Lịch sử Snapshots ---")
spark.sql("""
          SELECT committed_at, snapshot_id, operation
          FROM my_catalog.my_namespace.my_table.snapshots""").show()

--- Lịch sử Snapshots ---
+--------------------+-------------------+---------+
|        committed_at|        snapshot_id|operation|
+--------------------+-------------------+---------+
|2026-05-04 11:31:...|7628884848170549472|   append|
|2026-05-04 11:32:...|7192443500973530362|   append|
|2026-05-04 11:32:...|5331477788686933602|   append|
|2026-05-04 11:32:...|3541138966285469718|   delete|
|2026-05-04 11:32:...|3744620395294250159|   append|
+--------------------+-------------------+---------+



In [8]:
# Time Travel (tính năng của Data Lakehouse - xem lại dữ liệu tại một thời điểm trong quá khứ, điều mà Database không làm được)

# Cách 1: Dùng Snapshot ID (lấy từ bảng snapshots ở trên)

# Ví dụ xem tại thời điểm trước khi ChatGPT bị xóa
spark.read.option("snapshot-id", 5331477788686933602).table("my_catalog.my_namespace.my_table").sort('id').show()

+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Trung|        DE|
|  2|   Tuan|        BA|
|  3|    Duy|        AI|
|  4|   Thao|        DA|
|  5|ChatGPT|       LLM|
+---+-------+----------+



In [9]:
# Cách 2: Dùng mốc thời gian (Timestamp)

# Ví dụ truy vấn dữ liệu tại thời điểm 1 phút trước (hiện tại - 1 phút)
spark.sql("""
    SELECT * 
    FROM my_catalog.my_namespace.my_table 
    TIMESTAMP AS OF (current_timestamp() - INTERVAL 1 minutes)
    ORDER BY id
""").show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Trung|        DE|
|  2| Tuan|        BA|
+---+-----+----------+



In [10]:
# Schema Evolution (tính năng của Data Lakehouse - thêm/sửa/xóa cột thoải mái mà không lo làm hỏng dữ liệu)
 
# 1. Ví dụ Thêm cột mới
spark.sql("ALTER TABLE my_catalog.my_namespace.my_table ADD COLUMN (age int)")

# 2. Ví dụ Đổi tên cột (Từ department thành dept)
spark.sql("ALTER TABLE my_catalog.my_namespace.my_table RENAME COLUMN department TO dept")

# 3. Chèn dữ liệu mới để thấy sự thay đổi
spark.sql("INSERT INTO my_catalog.my_namespace.my_table VALUES (7, 'Claude', 'LLM', 2)")

# Kiểm tra kết quả
spark.sql("SELECT * FROM my_catalog.my_namespace.my_table ORDER BY id").show()

+---+------+----+----+
| id|  name|dept| age|
+---+------+----+----+
|  1| Trung|  DE|NULL|
|  2|  Tuan|  BA|NULL|
|  3|   Duy|  AI|NULL|
|  4|  Thao|  DA|NULL|
|  6|Gemini| LLM|NULL|
|  7|Claude| LLM|   2|
+---+------+----+----+

